In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import logging

LOGGER = logging.getLogger(__name__)

logging.basicConfig(level=logging.CRITICAL, format="%(name)s %(asctime)s %(message)s")
LOGGER.setLevel(logging.INFO)

In [48]:
from utils.database_utils import generate_database_and_retriever, populate_database
from scipy.spatial.distance import cosine

In [16]:
from utils.node_standarization import translate_nodes

In [19]:
data_base = "./localdb"
retriever = generate_database_and_retriever(main_folder=data_base)

all_keys = list(retriever.docstore.yield_keys())
all_documents = retriever.docstore.mget(all_keys)
docutments_dic = {all_keys[i]: all_documents[i] for i in range(len(all_keys))}

In [20]:
import pickle

with open("graph_raw_data.pkl", "rb") as f:
    graph_elements = pickle.load(f)

In [22]:
graph_elements_translated = translate_nodes(graph_elements)

In [25]:
graph_elements_translated

{'ad4f23557b79a1869da0347cd4f8532ac1232a6dd9bb6075a2cbcd4efe41c446': [Relationship(head='analysis', head_type='WORK_OF_ART', head_description='a qualitative understanding of classification errors', relation='PROVIDE', tail='understanding', tail_type='CONCEPT', tail_description='a qualitative understanding of classification errors', confidence=0.95, context='Los resultados del análisis anterior proporcionan una comprensión cualitativa de los errores de clasificación del modelo.', language='en'),
  Relationship(head='methodology', head_type='METHOD', head_description='a methodology described in section 4.3.3', relation='FOLLOWED_BY', tail='training', tail_type='PROCESS', tail_description='training two interpretable surrogate models', confidence=0.95, context='se ha seguido la metodología descrita en la sección 4.3.3 para entrenar dos modelos subrogados interpretables.', language='en'),
  Relationship(head='model', head_type='ORG', head_description='a linear regression model', relation='I

In [40]:
from utils.node_standarization import lemmatize_nodes_and_relationships

graph_elements_lematized = lemmatize_nodes_and_relationships(graph_elements_translated)

In [ ]:
from langchain_ollama import OllamaEmbeddings
from typing import List


class EmbeddingNode:
    model_name: str

    def __init__(self, model_name):
        self.model = OllamaEmbeddings(model=model_name)

    def get_embedding(self, text):
        return self.model.embed_query(text)


class Node:
    embedding: List[int]
    name: str
    type: str
    description: str
    model: EmbeddingNode

    def __init__(self, name, type, description, model):
        self.name = name
        self.type = type
        self.description = description
        self.model = model

    def get_embedding(self):
        self.embedding = self.model.get_embedding(
            self.name + " " + self.type + " " + self.description
        )


In [43]:
import tqdm

In [44]:
all_nodes = []
embedding_model = EmbeddingNode(model_name="embeddinggemma:latest")
for _, relations in tqdm.tqdm(
    graph_elements_translated.items(), total=len(graph_elements_translated)
):
    for relationship in relations:
        node_head = Node(
            name=relationship.head,
            type=relationship.head_type,
            description=relationship.head_description,
            model=embedding_model,
        )
        node_head.get_embedding()
        all_nodes.append(node_head)

        node_tail = Node(
            name=relationship.tail,
            type=relationship.tail_type,
            description=relationship.tail_description,
            model=embedding_model,
        )
        node_tail.get_embedding()
        all_nodes.append(node_tail)


100%|██████████| 9/9 [00:10<00:00,  1.19s/it]


In [ ]:
cosine([1, 2, 3], [1, 2, 3])

np.float64(0.0)

In [54]:
threshold = 0.1
for node in all_nodes:
    print(node.name)
    nodes_within_threshold = [
        node2.name
        for node2 in all_nodes
        if cosine(node.embedding, node2.embedding) < threshold
    ]
    print(nodes_within_threshold)

analysis
['analysis']
understand
['understand']
methodology
['methodology']
training
['training']
model
['model', 'model']
linear regression model
['linear regression model', 'regression model', 'linear regression', 'linear regression', 'linear regression', 'linear regression', 'linear regression', 'linear regression', 'linear regression model', 'linear regression', 'linear regression', 'linear regression', 'linear regression', 'linear regression', 'linear regression']
model
['model', 'model']
decision tree model
['decision tree model', 'decision tree model', 'decision tree', 'decision tree', 'decision tree', 'decision tree', 'decision tree', 'decision tree', 'decision tree', 'decision tree', 'decision tree', 'decision tree', 'decision tree', 'decision tree', 'decision tree model', 'decision tree', 'decision tree', 'decision tree', 'decision tree', 'decision tree', 'decision tree']
model
['model']
prediction
['prediction']
model
['model', 'model']
roc curve
['roc curve', 'roc curve']
m

In [ ]:
all_nodes

In [17]:
import pickle

with open("graph_lematized_data.pkl", "wb") as f:
    pickle.dump(graph_elements_lematized, f)